# Zajęcie 1 — demo: Regresja liniowa i klasyfikacja binarna (SVM)

Notebook jest przygotowany jako **szablon do zajęć**: zawiera kompletny przepływ pracy ML (definicja problemu → dane → ocena → cechy → modelowanie → eksperymenty) oraz dwie ścieżki:
- **Regresja liniowa wielowymiarowa**
- **Klasyfikacja binarna SVM**

Materiały merytoryczne odpowiadają opisom z instrukcji do zajęć (regresja liniowa, SVM, kroki workflow).

## 0. Wymagania i uruchomienie
1. Umieść plik danych `CSV` w tym samym folderze co notebook **albo** podaj ścieżkę do pliku.
2. Zainstaluj zależności (jeśli trzeba): `pandas`, `numpy`, `matplotlib`, `scikit-learn`.

> Warianty danych w instrukcji to m.in. Parkinson, palący vs niepalący, powikłania zawału, kardiotokografia itp. Notebook jest **uniwersalny** — działa dla dowolnego CSV po wskazaniu kolumny docelowej.

In [ ]:
# Jeśli używasz środowiska bez bibliotek, odkomentuj:
# !pip -q install pandas numpy matplotlib scikit-learn

## 1. Importy i ustawienia

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score
)

np.random.seed(42)

## 2. Konfiguracja zadania (podmień pod swój wariant)
Ustaw:
- `csv_path` — ścieżka do pliku CSV
- `target` — nazwa kolumny z etykietą/zmienną zależną
- `task_type` — `"regression"` albo `"classification"`

**Wskazówka:** jeśli w danych jest kolumna typu `object`/`string` z klasą (np. `yes/no`), notebook sam spróbuje zakodować ją do 0/1 dla klasyfikacji.

In [3]:
#classification
# po class tzn czy jest chory czy nie
path=r"C:\Users\USER098\Documents\GitHub\Artifical_Inteligence_lesson\Data\pd_speech_features.csv"
targets="class"

@dataclass
class Config:
    csv_path: str = path    # <- podmień
    target: str = targets      # <- podmień
    task_type: str = "classification"  # "regression" lub "classification"
    test_size: float = 0.2
    random_state: int = 42

cfg = Config()
cfg

Config(csv_path='C:\\Users\\USER098\\Documents\\GitHub\\Artifical_Inteligence_lesson\\Data\\pd_speech_features.csv', target='class', task_type='classification', test_size=0.2, random_state=42)

In [3]:
#najlepszy parametr pod regresje
path=r"C:\Users\USER098\Documents\GitHub\Artifical_Inteligence_lesson\Data\pd_speech_features.csv"
df = pd.read_csv(path)

corr = df.corr(numeric_only=True)
target = corr.drop("id").abs().sum().sort_values(ascending=False).index[0]
print(target)

tqwt_entropy_log_dec_13


In [ ]:
#regression
# po PPE tzn Pitch Period Entropy mierzy nieregularność głosu często używana w badaniach Parkinsona
# lub TQWT – Tunable Q-factor Wavelet Transform entropy_log – entropia logarytmiczna sygnału dec_13 – 13 poziom dekompozycji
# miara złożoności sygnału głosu w jednym z pasm częstotliwości.
path=r"C:\Users\USER098\Documents\GitHub\Artifical_Inteligence_lesson\Data\pd_speech_features.csv"
targets="PPE"

@dataclass
class Config:
    csv_path: str = path    # <- podmień
    target: str = targets      # <- podmień
    task_type: str = "regression"  # "regression" lub "classification"
    test_size: float = 0.2
    random_state: int = 42

cfg = Config()
cfg

## 3. Wczytanie danych

In [4]:
def load_data(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Nie znaleziono pliku: {path}. "
            "Podmień cfg.csv_path na poprawną ścieżkę lub nazwę pliku."
        )
    df = pd.read_csv(path)
    return df

df = load_data(cfg.csv_path)
df.head()

,id,gender,PPE,DFA,RPDE,numPulses,numPeriodsPulses,meanPeriodPulses,stdDevPeriodPulses,locPctJitter,...,tqwt_kurtosisValue_dec_28,tqwt_kurtosisValue_dec_29,tqwt_kurtosisValue_dec_30,tqwt_kurtosisValue_dec_31,tqwt_kurtosisValue_dec_32,tqwt_kurtosisValue_dec_33,tqwt_kurtosisValue_dec_34,tqwt_kurtosisValue_dec_35,tqwt_kurtosisValue_dec_36,class
0,0,1,0.85247,0.71826,0.57227,240,239,0.008064,0.000087,0.00218,...,1.5620,2.6445,3.8686,4.2105,5.1221,4.4625,2.6202,3.0004,18.9405,1
1,0,1,0.76686,0.69481,0.53966,234,233,0.008258,0.000073,0.00195,...,1.5589,3.6107,23.5155,14.1962,11.0261,9.5082,6.5245,6.3431,45.1780,1
2,0,1,0.85083,0.67604,0.58982,232,231,0.008340,0.000060,0.00176,...,1.5643,2.3308,9.4959,10.7458,11.0177,4.8066,2.9199,3.1495,4.7666,1
3,1,0,0.41121,0.79672,0.59257,178,177,0.010858,0.000183,0.00419,...,3.7805,3.5664,5.2558,14.0403,4.2235,4.6857,4.8460,6.2650,4.0603,1
4,1,0,0.32790,0.79782,0.53028,236,235,0.008162,0.002669,0.00535,...,6.1727,5.8416,6.0805,5.7621,7.7817,11.6891,8.2103,5.0559,6.1164,1


### Podstawowe informacje o danych

In [11]:
display(df.shape)
# display(df.dtypes.head(20))
print(df.dtypes.head(20))
# display(df.describe(include='all').T.head(5))
print(df.describe(include='all').T.head(5))

(756, 755)

id                      int64
gender                  int64
PPE                   float64
DFA                   float64
RPDE                  float64
numPulses               int64
numPeriodsPulses        int64
meanPeriodPulses      float64
stdDevPeriodPulses    float64
locPctJitter          float64
locAbsJitter          float64
rapJitter             float64
ppq5Jitter            float64
ddpJitter             float64
locShimmer            float64
locDbShimmer          float64
apq3Shimmer           float64
apq5Shimmer           float64
apq11Shimmer          float64
ddaShimmer            float64
dtype: object
        count        mean        std       min        25%         50%  \
id      756.0  125.500000  72.793721  0.000000  62.750000  125.500000   
gender  756.0    0.515873   0.500079  0.000000   0.000000    1.000000   
PPE     756.0    0.746284   0.169294  0.041551   0.762833    0.809655   
DFA     756.0    0.700414   0.069718  0.543500   0.647053    0.700525   
RPDE    756.0    0.48

## 4. Podział na X / y i wstępne czyszczenie
Wg przepływu pracy ML w instrukcji najpierw definiujemy problem i dane, a następnie przechodzimy do cech i modelowania.

W tym notebooku:
- usuwamy wiersze bez wartości w `target`
- rozdzielamy cechy numeryczne i kategoryczne
- budujemy `Pipeline`, który:
  - imputuje braki (median/most_frequent)
  - koduje kategorie (`OneHotEncoder`)
  - skaluje cechy numeryczne (`StandardScaler`) — ważne m.in. dla SVM

In [5]:
df = df.copy()

if cfg.target not in df.columns:
    raise ValueError(f"Kolumna target='{cfg.target}' nie istnieje. Dostępne kolumny: {list(df.columns)[:50]} ...")

df = df.dropna(subset=[cfg.target])

X = df.drop(columns=[cfg.target])
y = df[cfg.target]

# Prosta konwersja etykiet tekstowych do 0/1 w klasyfikacji binarnej
if cfg.task_type == "classification" and y.dtype == "object":
    y = y.astype(str).str.strip().str.lower()
    uniq = sorted(y.unique())
    if len(uniq) == 2:
        mapping = {uniq[0]: 0, uniq[1]: 1}
        y = y.map(mapping)
        print("Zmapowano etykiety:", mapping)
    else:
        raise ValueError(f"Klasyfikacja binarna wymaga 2 klas, a wykryto: {uniq}")

# Rozpoznanie typów cech
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = [c for c in X.columns if c not in num_features]

num_features[:10], cat_features[:10], len(num_features), len(cat_features)

(['id',
  'gender',
  'PPE',
  'DFA',
  'RPDE',
  'numPulses',
  'numPeriodsPulses',
  'meanPeriodPulses',
  'stdDevPeriodPulses',
  'locPctJitter'],
 [],
 754,
 0)

## 5. Podział train/test
- Regresja: zwykły `train_test_split`
- Klasyfikacja: `stratify=y` (utrzymanie proporcji klas)

In [6]:
stratify = y if cfg.task_type == "classification" else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=cfg.test_size,
    random_state=cfg.random_state,
    stratify=stratify
)

X_train.shape, X_test.shape

((604, 754), (152, 754))

## 6. Preprocessing jako `ColumnTransformer` + `Pipeline`

In [7]:
#skonczylem
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ],
    remainder="drop"
)

preprocess

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['id', 'gender', 'PPE', 'DFA', 'RPDE',
                                  'numPulses', 'numPeriodsPulses',
                                  'meanPeriodPulses', 'stdDevPeriodPulses',
                                  'locPctJitter', 'locAbsJitter', 'rapJitter',
                                  'ppq5Jitter', 'ddpJitter', 'locShimmer',
                                  'locDbShimmer', 'apq3Shimmer', 'apq5Shimmer',
                                  'apq11Shimmer', 'ddaShimmer',
                                  'meanAutoCorrHarmonicity',
                                  'meanNoiseToHarmHarmonicity',
                                  'meanHarmToNoiseHarmonicity', 'minIntensity',
                                  'maxIntensity', 'meanIntensity', 'f1', 'f2',
                                  'f3', 'f4', ...]),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 [])])

## 7A. Ścieżka 1 — Regresja liniowa (wielowymiarowa)
W instrukcji omawiana jest postać macierzowa oraz estymacja współczynników (równania normalne) dla regresji liniowej wielowymiarowej.

Tutaj używamy `scikit-learn`, ale idea jest ta sama: dopasowanie współczynników minimalizujących sumę kwadratów błędów.

**Ocena**: MSE, MAE, R².

In [8]:
def run_regression():
    # Modele do porównania (bazowy + regularizacja)
    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0),
        "Lasso": Lasso(alpha=0.001, max_iter=5000)
    }

    results = []
    fitted = {}

    for name, model in models.items():
        pipe = Pipeline(steps=[("preprocess", preprocess),
                              ("model", model)])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        mse = mean_squared_error(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        results.append({"model": name, "MSE": mse, "MAE": mae, "R2": r2})
        fitted[name] = pipe

    res_df = pd.DataFrame(results).sort_values("MSE")
    return res_df, fitted

# Uruchom tylko jeśli wybrano regresję:
if cfg.task_type == "regression":
    res_df, fitted_models = run_regression()
    display(res_df)


### Wizualizacja: predykcja vs prawda (regresja)

In [10]:
if cfg.task_type == "regression":
    best_name = res_df.iloc[0]["model"]
    best = fitted_models[best_name]
    preds = best.predict(X_test)

    plt.figure(figsize=(6,6))
    plt.scatter(y_test, preds, alpha=0.6)
    plt.xlabel("y (prawda)")
    plt.ylabel("y_hat (predykcja)")
    plt.title(f"Regresja — {best_name}: y vs y_hat")
    plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)])
    plt.show()


### Walidacja krzyżowa (regresja)
W instrukcji jako element oceny wspomniana jest walidacja krzyżowa (cross-validation).

In [11]:
if cfg.task_type == "regression":
    # 5-fold CV na najlepszym modelu
    best_name = res_df.iloc[0]["model"]
    best = fitted_models[best_name]

    cv = KFold(n_splits=5, shuffle=True, random_state=cfg.random_state)
    scores = cross_val_score(best, X, y, cv=cv, scoring="r2")
    print("R2 CV (mean ± std):", scores.mean(), "±", scores.std())


## 7B. Ścieżka 2 — Klasyfikacja binarna SVM
W instrukcji opisano SVM, margines i funkcję decyzyjną oraz rolę hiperparametru `C` i jąder (kernels).

Tutaj:
- używamy `SVC` (SVM z jądrem)
- wykonujemy prosty `GridSearchCV` po `C`, `kernel` oraz `gamma`

**Ocena**: accuracy, precision, recall, F1, AUC (jeśli dostępne).

In [12]:
def run_svm_classification():
    # Bazowy model + siatka hiperparametrów (mała, pod demo)
    svm = SVC(probability=True)

    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", svm)
    ])

    param_grid = {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg.random_state)

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1
    )

    search.fit(X_train, y_train)
    return search

if cfg.task_type == "classification":
    search = run_svm_classification()
    print("Najlepsze parametry:", search.best_params_)
    print("Najlepszy wynik CV (F1):", search.best_score_)


Najlepsze parametry: {'model__C': 10, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Najlepszy wynik CV (F1): 0.9181831558017925


### Ewaluacja na zbiorze testowym (SVM)

In [ ]:
if cfg.task_type == "classification":
    best = search.best_estimator_
    y_pred = best.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title("Macierz pomyłek — SVM")
    plt.show()

    # ROC/AUC (wymaga prawdopodobieństw lub decision_function)
    if hasattr(best.named_steps["model"], "predict_proba"):
        y_score = best.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_score)
        print(f"AUC={auc:.4f}")
        RocCurveDisplay.from_predictions(y_test, y_score)
        plt.title("ROC — SVM")
        plt.show()


## 8. Podsumowanie i co pokazać na zajęciach
Proponowany scenariusz demo (10–25 min):
1. Wczytanie danych, szybki `head()/describe()`
2. Wybór `target` + ustawienie `task_type`
3. Podział train/test (z `stratify` dla klasyfikacji)
4. Pipeline preprocessingu (imputer + one-hot + scaler)
5. Model:
   - regresja: Linear/Ridge/Lasso + metryki MSE/MAE/R²
   - klasyfikacja: SVM + GridSearchCV + macierz pomyłek + ROC/AUC
6. (Opcjonalnie) walidacja krzyżowa.

W razie potrzeby można szybko rozszerzyć notebook o selekcję cech, balansowanie klas (np. `class_weight='balanced'`), albo porównanie z logistyczną regresją / drzewami.

## 9. Awaryjnie: dane syntetyczne (gdy nie ma CSV)
Jeżeli na demo nie masz jeszcze gotowego zbioru, możesz uruchomić poniższą komórkę, aby wygenerować mały zbiór do klasyfikacji binarnej i przetestować SVM.

*Uwaga*: to tylko opcja awaryjna, nie zastępuje wariantu z instrukcji.

In [ ]:
# Generowanie danych syntetycznych (opcjonalnie) — uruchom TYLKO gdy nie masz CSV.
# Komórka zapisze plik synthetic.csv i ustawi cfg pod klasyfikację.

from sklearn.datasets import make_classification

def make_synthetic_csv(path="synthetic.csv"):
    X_syn, y_syn = make_classification(
        n_samples=800,
        n_features=12,
        n_informative=6,
        n_redundant=2,
        n_clusters_per_class=2,
        class_sep=1.2,
        flip_y=0.03,
        random_state=42
    )
    df_syn = pd.DataFrame(X_syn, columns=[f"f{i}" for i in range(X_syn.shape[1])])
    df_syn["target"] = y_syn
    df_syn.to_csv(path, index=False)
    return path

# odkomentuj jeśli potrzebujesz:
# path = make_synthetic_csv()
# cfg.csv_path = path
# cfg.target = "target"
# cfg.task_type = "classification"
# print("Zapisano:", path, "— możesz wrócić do sekcji 3 i uruchomić notebook od wczytania danych.")
